<a href="https://colab.research.google.com/github/soumya24sahu/-ML-Based-Equalizer-for-BER-Improvement-in-Long-Haul-Optical-Fiber-Communication/blob/main/data_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install OptiCommPy   # install opticommpy python library

In [ ]:
import numpy as np
from optic.dsp.core import pulseShape, firFilter, decimate, symbolSync, pnorm, signalPower, upsample
from optic.models.devices import pdmCoherentReceiver, basicLaserModel, coherentReceiver
from optic.models.tx import simpleWDMTx
from optic.utils import parameters, dBm2W
from optic.models.channels import ssfm
from optic.comm.metrics import fastBERcalc, monteCarloGMI, monteCarloMI, calcEVM
from optic.plot import pconst, plotPSD
from optic.dsp.equalization import edc
import matplotlib.pyplot as plt
import tensorflow as tf

In [ ]:
# Set the seed for NumPy's random number generator so that any random
# operations that follow are reproducible
np.random.seed(seed=123)
size = 2e5

batch_size = int(2e5)   # Define batch_size as an integer version of 2e5 (200,000)


# Create a 1D array of zeros with length (size * 2) = 400,000 elements
# dtype=complex64 means each element is a complex number using
# 32-bit floats for both the real and imaginary parts (lower precision,
# less memory than the default complex128)
data = np.zeros((int(size*2),), dtype=np.complex64)

# Create a 2D array of zeros with shape (400,000, 1) — i.e., a column vector
# Also complex64, matching the data array's dtype
# This is likely meant to hold a label/target value for each entry in `data`
label = np.zeros((int(size*2),1), dtype=np.complex64)

In [ ]:
# Transmitter parameters:
paramTx = parameters()
paramTx.M   = 16           # order of the modulation format
paramTx.Rs  = 10e9         # symbol rate [baud]
paramTx.SpS = 16           # samples per symbol
paramTx.pulseType = 'rrc'      # pulse shaping filter
paramTx.nFilterTaps = 1024     # number of pulse shaping filter coefficients
paramTx.pulseRollOff = 0.01    # RRC rolloff
paramTx.powerPerChannel = 3        # power per WDM channel [dBm]
paramTx.nChannels = 1       # number of WDM channels
paramTx.Fc      = 193.1e12 # central optical frequency of the WDM spectrum
paramTx.laserLinewidth  = 10e3    # laser linewidth in Hz
paramTx.wdmGridSpacing  = 37.5e9  # WDM grid spacing
paramTx.nPolModes = 1         # number of signal modes [2 for polarization multiplexed signals]
paramTx.seed   = 123
paramTx.nBits = int(np.log2(paramTx.M)*batch_size) # total number of bits per polarization


In [ ]:
# fiber parameter
Num_OF = 15                     # Number of optical fibers
OF_Len = 80                     # Optical Fiber Length [km]
OF_loss = 0.2                   # Optical fiber loss
samp_freq = paramTx.Rs * paramTx.SpS   #Sampling Frequency

# non linear fiber optical channel parameters
paramCh = parameters()
paramCh.Ltotal = Num_OF * OF_Len  # total length [km]
paramCh.Lspan = OF_Len            # total link distance [km]
paramCh.hz = 10                  # step size [km]
paramCh.alpha = OF_loss           # fiber loss parameter [dB/km]
paramCh.gamma = 1.3               # fiber non linear parameter 1/W/km
paramCh.D = 16                    # fiber dispersion parameter [ps/nm/km]
paramCh.Fc = 193.1e12             # central optical frequency [Hz]
paramCh.Fs = samp_freq   #Sampling Frequency
paramCh.amp = 'edfa'             #Optical Amplifier
paramCh.NF = 5                  #Noise Factor in dB
paramCh.prec   = np.complex128
paramCh.prgsBar = True


In [ ]:
Fc = paramTx.Fc
Ts = 1/samp_freq

# local oscillator (LO) parameters:
FO  = 0
 #150e6
 # frequency offset

# generate CW laser LO field
paramLO = parameters()
paramLO.P = 10              # power in dBm
paramLO.lw = 100e3          # laser linewidth
paramLO.RIN_var = 1e-20
paramLO.Ns = int(paramTx.SpS * batch_size)
paramLO.Fs = samp_freq

# photodiodes parameters
paramPD = parameters()
paramPD.B = 1 * paramTx.Rs
paramPD.Fs = samp_freq
paramPD.ideal = False



# size // batch_size = 1, so this loop actually only runs once.
# (This structure suggests the code was designed to be flexible for cases where size > batch_size, generating data in multiple chunks.)
for i in range(int(size // batch_size)):

    # Generate a WDM optical signal. This simulates multiple optical channels (wavelengths) each carrying modulated data, combined into a single transmitted signal.
    # Returns:
    #   sigWDM_Tx : the transmitted WDM signal (time-domain waveform)
    #   symbTx_   : the transmitted symbols (before pulse shaping/modulation)
    #   paramTx   : (possibly updated) transmitter parameters
    sigWDM_Tx, symbTx_, paramTx = simpleWDMTx(paramTx)

    # symbTx_ likely has shape (N_symbols, N_channels, N_polarizations)
    # or similar. Here we select only the first polarization/dimension
    # (index 0 along the last axis), discarding the rest.
    symbTx_ = symbTx_[:, :, 0]

    # CHANNEL (FIBER PROPAGATION)
    # Propagate the WDM signal through an optical fiber channel using the SSFM (Split-Step Fourier Method)
    sigCh = ssfm(sigWDM_Tx, paramCh)

    #  RECEIVER SIDE
    # Generate the Local Oscillator (LO) laser signal, used for coherent detection at the receiver
    sigLO = basicLaserModel(paramLO)

    # Simulate a coherent receiver: mixes the received channel signal(sigCh) with the local oscillator (sigLO) using a photodiode (PD)
    # model, converting the optical signal back into an electrical signal that captures amplitude and phase
    sigRx = coherentReceiver(sigCh, sigLO, paramPD)

    # DOWNSAMPLING
    # The received signal is oversampled  Here it's downsampled by taking  every 8th sample, presumably because the simulation used 8 samples
    # per symbol (oversampling factor = 8), reducing back to ~1 sample  per symbol
    sigDe = sigRx[::8]

    #  STORE RESULTS INTO PREALLOCATED ARRAYS
    # Store the downsampled received signal into the `data` array. Since this is the i-th batch, it's placed into the corresponding slice of the array.
    # The "*2" suggests each batch produces ,2 * batch_size samples (e.g., 2 polarizations, or 2 WDM channels, flattened together into one array).
    data[i*batch_size*2 : (i+1)*batch_size*2] = sigDe

    # Store the corresponding transmitted signal (also downsampled by 8)
    label[i*batch_size*2:(i+1)*batch_size*2] = sigWDM_Tx[::8]

In [ ]:
print(data)
print(label)

In [ ]:
# plot real part of tx and rx 500 data without downsampling
import matplotlib.pyplot as plt

# Extract the real part (in-phase, "I" component) of `sigCh`
real_data = sigCh.real

# Extract the imaginary part (quadrature, "Q" component) of `sigCh`
imag_data = sigCh.imag

# Extract the real part of `sigWDM_Tx`
real_label = sigWDM_Tx.real

# Extract the imaginary part of `sigWDM_Tx`
imag_label = sigWDM_Tx.imag

plt.figure(figsize=(10,5))
plt.plot(real_data[:500], label="Real Rx", alpha=0.8)
plt.plot(real_label[:500], label="Real Tx", alpha=0.8)
plt.xlabel("Sample Index")
plt.ylabel("Amplitude")
plt.title("Real Part: Tx vs Rx without downsampling")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# plot real part of tx and rx 500 data after downsampling
real_data = data.real
imag_data = data.imag
real_label = label.real
imag_label = label.imag
plt.figure(figsize=(10,5))
plt.plot(real_data[:500], label="Real Rx", alpha=0.8)
plt.plot(real_label[:500], label="Real Tx", alpha=0.8)

plt.xlabel("Sample Index")
plt.ylabel("Amplitude")
plt.title("Real Part: Tx vs Rx")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# plot psd
Fs = paramCh.Fs
fig,_ = plotPSD(label, Fs, paramCh.Fc, label='Tx');
fig, ax = plotPSD(data, Fs, paramCh.Fc, fig=fig, label='Rx');
fig.set_figheight(3)
fig.set_figwidth(10)
ax.set_title('optical WDM spectrum');

In [ ]:

from google.colab import drive

# Mount Google Drive to the Colab virtual machine's filesystem at '/content/drive'
drive.mount('/content/drive')

# Import the os module for filesystem operations (creating directories, etc.)
import os

data = data[:int(2.2e7)]   # Truncate `data` to only the first 22,000,000 (2.2e7) elements , if smaller than this it has no effect
label = label[:int(2.2e7)]  # Truncate `label` the same way, keeping it aligned with `data`

# Define the target directory path in Google Drive where the files will be saved
path = "/content/drive/MyDrive/Project/test/10_1200_new"

# Create the directory (and any missing parent directories) if it doesn't already exist
os.makedirs(path, exist_ok=True)

# Save `data` to disk as a .npy file, after normalizing it:
# all values into the range [-1, 1] (in terms of magnitude) — a common
# preprocessing step before feeding data into a machine learning model
np.save(path + '/data.npy', data/np.max(np.abs(data)))

# Save `label` similarly, normalized by its own maximum absolute value
np.save(path + '/label.npy', label/np.max(np.abs(label)))

In [ ]:
# generate CW laser LO field
paramLO = parameters()
paramLO.P = 10              # power in dBm
paramLO.lw = 100e3          # laser linewidth
paramLO.RIN_var = 0
paramLO.Ns = int(paramTx.SpS * batch_size)
paramLO.Fs = Fs

# photodiodes parameters
paramPD = parameters()
paramPD.B = 2 * paramTx.Rs
paramPD.Fs = Fs
paramPD.ideal = True

# Decimate
paramDec = parameters()
paramDec.SpSin  = paramTx.SpS
paramDec.SpSout = 2

# Receiver parameters
sigLO = basicLaserModel(paramLO)
sigRx = coherentReceiver(Ech, sigLO ,paramPD)

print(sigRx)

# decimate the recived signal

sigRx_down = decimate(sigRx, paramDec)
#sigRx_down= sigRx[0::7]
print(sigRx_down)


 # plot psd of tx and rx after downsample
Fs = paramCh.Fs
fig,_ = plotPSD(sigWDM_Tx, Fs, paramCh.Fc, label='Tx');
fig, ax = plotPSD(sigRx, Fs, paramCh.Fc, fig=fig, label='Rx_downS');
fig.set_figheight(3)
fig.set_figwidth(10)
ax.set_title('optical WDM spectrum after down sampling');



In [ ]:
# plot of received signal right after matched filtering, before any correction
data = sigRx
label = sigWDM_Tx

from google.colab import drive
drive.mount('/content/drive')

data = data[:int(2.2e7)]
label= label[:int(2.2e7)]
path = "/content/drive/MyDrive/Project/test"
np.save(path + '/data2.npy', data/np.max(np.abs(data)))
np.save(path + '/label2.npy', label/np.max(np.abs(label)))


paramPulse = parameters()
paramPulse.pulseType   = 'rrc'
paramPulse.SpS         = paramTx.SpS          # same SpS as TX
paramPulse.nFilterTaps = paramTx.nFilterTaps        # same # of taps
paramPulse.rollOff   = paramTx.pulseRollOff     # roll-off

pulse = pulseShape(paramPulse)

sigRx = firFilter(pulse, Ech)
sym_rx = sigRx[0::8]
print(sym_rx[:10])

plt.figure(figsize=(5,5))
plt.scatter(sym_rx.real, sym_rx.imag, s=2, alpha=0.4)
plt.axis('equal'); plt.grid(True)
plt.title('RX Constellation (aligned)')
plt.show()



In [ ]:
# plot of received signal after  the least-squares gain/phase correction

N = min(len(sym_tx), len(sym_rx))
g = np.vdot(sym_tx[:N], sym_rx[:N]) / np.vdot(sym_rx[:N], sym_rx[:N])
sym_al = g * sym_rx[:N]
print(sym_al[:10])  # should now be around ±0.316 and ±0.949
plt.figure(figsize=(5,5))
plt.scatter(sym_al.real, sym_al.imag, s=2, alpha=0.4)
plt.axis('equal'); plt.grid(True)
plt.title('RX Constellation (aligned)')
plt.show()